In [1]:
import polars as pl
import pandera.polars as pa
from datetime import datetime
from src.ecommerce_etl import config
from src.ecommerce_etl.io import read_csv_raw, write_parquet
from src.ecommerce_etl.transforms import (
    parse_datetime, strip_strings, lowercase, uppercase,
    cast_float, nullify_outside_range, nullify_not_in_set, split_orphans,
    drop_null_keys, deduplicate_by_key,
)

Customers

In [2]:
customers = read_csv_raw("customers")
s_customers = customers.sample(fraction=0.1, seed=42)
s_customers.head(5)

customer_id,customer_name,email,city,state,created_at
str,str,str,str,str,str
"""CUST-00001""","""Beatriz Pereira""","""beatriz.pereira_1@email.com""","""Salvador""","""BA""","""2022-05-14 01:01:00"""
"""CUST-00003""","""Diego Costa""","""diego.costa_3@email.com""","""Campinas""","""SP""","""2022-01-05 11:38:00"""
"""CUST-00004""","""Patricia Cardoso""","""patricia.cardoso_4@email.com""","""Belem""","""PA""","""2022-01-10 18:12:00"""
"""CUST-00008""","""Larissa Araujo""","""larissa.araujo_8@email.com""","""Caruaru""","""PE""","""2022-05-27 18:25:00"""
"""CUST-00006""","""Beatriz Oliveira""","""beatriz.oliveira_6@email.com""","""Goiania""","""GO""","""2023-08-03 22:35:00"""


In [3]:
class Customers(pa.DataFrameModel):
    customer_id: str = pa.Field(nullable=False, unique=True)
    customer_name: str = pa.Field(nullable=True)
    email: str = pa.Field(nullable=True)
    city: str = pa.Field(nullable=True)
    state: str = pa.Field(nullable=True)
    created_at: datetime = pa.Field(nullable=False)
    source_file: str = pa.Field(alias="_source_file", nullable=False)
    ingested_at: datetime = pa.Field(alias="_ingested_at", nullable=False)

    class Config:
        coerce = False
        strict = True

In [4]:
customers_bronze = s_customers.with_columns(
    pl.lit("customers.csv").alias("_source_file"),
    pl.lit(datetime.now()).alias("_ingested_at")
)
write_parquet(customers_bronze, config.BRONZE_DIR / "customers.parquet")

In [5]:
STRING_COLS = ["customer_id", "customer_name", "email", "city", "state"]

customers = parse_datetime(customers_bronze, "created_at")
customers = strip_strings(customers, STRING_COLS)
customers = lowercase(customers, "email")
customers = uppercase(customers, "state")

customers, rej_null = drop_null_keys(customers, "customer_id")
customers, rej_dup = deduplicate_by_key(customers, "customer_id", "created_at")

customers_silver = customers
customers_rejected = pl.concat([
    rej_null.with_columns(pl.lit("null_customer_id").alias("reject_reason")),
    rej_dup.with_columns(pl.lit("duplicate_customer_id").alias("reject_reason")),
], how="diagonal")

write_parquet(customers_silver, config.SILVER_DIR / "customers.parquet")
write_parquet(customers_rejected, config.QUARANTINE_DIR / "customers.parquet")

In [6]:
quality_rows = []
def record_check(check_name, table, column, checked, failed, stage):
    quality_rows.append({
        "check_name": check_name, "table": table, "column": column,
        "records_checked": checked, "records_failed": failed,
        "pct_failed": round(100 * failed / checked, 2) if checked else 0.0,
        "stage": stage, "executed_at": datetime.now(),
    })

null_counts = customers_bronze.select(["customer_name", "email", "city", "state"]).null_count()
for col in ["customer_name", "email", "city", "state"]:
    record_check("null_check", "customers", col, customers_bronze.height, null_counts[col][0], "bronze")
record_check("null_check", "customers", "customer_id", customers_bronze.height, rej_null.height, "bronze")
record_check("duplicate_check", "customers", "customer_id", customers_bronze.height, rej_dup.height, "bronze")
record_check("row_count", "customers", "*", customers_bronze.height, customers_bronze.height - customers_silver.height, "silver")

quality = pl.DataFrame(quality_rows)
write_parquet(quality, config.QUALITY_DIR / "customers.parquet")
quality

check_name,table,column,records_checked,records_failed,pct_failed,stage,executed_at
str,str,str,i64,i64,f64,str,datetime[μs]
"""null_check""","""customers""","""customer_name""",500,9,1.8,"""bronze""",2026-07-10 08:45:29.811382
"""null_check""","""customers""","""email""",500,16,3.2,"""bronze""",2026-07-10 08:45:29.811389
"""null_check""","""customers""","""city""",500,6,1.2,"""bronze""",2026-07-10 08:45:29.811391
"""null_check""","""customers""","""state""",500,1,0.2,"""bronze""",2026-07-10 08:45:29.811392
"""null_check""","""customers""","""customer_id""",500,0,0.0,"""bronze""",2026-07-10 08:45:29.811422
"""duplicate_check""","""customers""","""customer_id""",500,0,0.0,"""bronze""",2026-07-10 08:45:29.811463
"""row_count""","""customers""","""*""",500,0,0.0,"""silver""",2026-07-10 08:45:29.811476


In [7]:
try:
    Customers.validate(customers_silver, lazy=True)
    print("Customers silver table passed")
except pa.errors.SchemaErrors as e:
    display(e.failure_cases)

Customers silver table passed


Products

In [8]:
products = read_csv_raw("products")
s_products = products.sample(fraction=0.1, seed=42)
s_products.head(5)

product_id,product_name,category,price,weight_kg
str,str,str,str,str
"""PROD-0023""","""Office Supplies Item 23""","""office_supplies""","""4131.69""","""11.46"""
"""PROD-0024""","""Books Item 24""","""books""","""4123.64""",null
"""PROD-0067""","""Home Appliances Item 67""","""home_appliances""","""103.3""","""19.7"""
"""PROD-0075""","""Home Appliances Item 75""","""home_appliances""","""3558.07""","""36.67"""
"""PROD-0082""","""Furniture Item 82""",null,"""2470.33""","""20.06"""


In [9]:
class Products(pa.DataFrameModel):
    product_id: str = pa.Field(nullable=False, unique=True)
    product_name: str = pa.Field(nullable=True)
    category: str = pa.Field(nullable=True, isin=list(config.PRODUCT_CATEGORIES))
    price: float = pa.Field(nullable=True, ge=0)
    weight_kg: float = pa.Field(nullable=True, ge=0)
    source_file: str = pa.Field(alias="_source_file", nullable=False)
    ingested_at: datetime = pa.Field(alias="_ingested_at", nullable=False)

    class Config:
        coerce = False
        strict = True

In [10]:
products_bronze = s_products.with_columns(
    pl.lit("products.csv").alias("_source_file"),
    pl.lit(datetime.now()).alias("_ingested_at")
)
write_parquet(products_bronze, config.BRONZE_DIR / "products.parquet")

### Explorar categorías → derivar el set de valores aceptados

Antes de validar `category` contra una lista de valores permitidos, hay que **saber cuáles son**. Aquí exploramos las categorías presentes en el crudo y las enumeramos.

**Suposición:** derivar la lista directamente de los datos **de esta forma** solo es válido porque este dataset es **pequeño y fijo**, así que se pueden ver todas fácilmente. En un caso real la lista de categorías válidas debería venir de las **reglas de negocio**, de una **tabla de dimensión** o de una exploración mas optimizada. La lista final (derivada de esta exploración) vive en `config.PRODUCT_CATEGORIES` y se usa en el `accepted_values` check.

In [11]:
categorias = sorted(read_csv_raw("products")["category"].drop_nulls().unique().to_list())
print(f"{len(categorias)} categorías encontradas:")
categorias

15 categorías encontradas:


['automotive',
 'beauty',
 'books',
 'clothing',
 'electronics',
 'food',
 'furniture',
 'garden',
 'health',
 'home_appliances',
 'music',
 'office_supplies',
 'pet_shop',
 'sports',
 'toys']

In [12]:
products = strip_strings(products_bronze, ["product_id", "product_name", "category"])
products = lowercase(products, "category")
products, cast_price = cast_float(products, "price")
products, cast_weight = cast_float(products, "weight_kg")
products, neg_price = nullify_outside_range(products, "price", low=0)
products, rej_cat = nullify_not_in_set(products, "category", config.PRODUCT_CATEGORIES)

products, rej_null = drop_null_keys(products, "product_id")
products, rej_dup = deduplicate_by_key(products, "product_id")

products_silver = products
products_rejected = pl.concat([
    rej_null.with_columns(pl.lit("null_product_id").alias("reject_reason")),
    rej_dup.with_columns(pl.lit("duplicate_product_id").alias("reject_reason")),
    rej_cat.with_columns(pl.lit("invalid_category").alias("reject_reason")),
], how="diagonal")

write_parquet(products_silver, config.SILVER_DIR / "products.parquet")
write_parquet(products_rejected, config.QUARANTINE_DIR / "products.parquet")

In [13]:
quality_rows = []
def record_check(check_name, table, column, checked, failed, stage):
    quality_rows.append({
        "check_name": check_name, "table": table, "column": column,
        "records_checked": checked, "records_failed": failed,
        "pct_failed": round(100 * failed / checked, 2) if checked else 0.0,
        "stage": stage, "executed_at": datetime.now(),
    })

null_counts = products_bronze.select(["product_name", "category", "price", "weight_kg"]).null_count()
for col in ["product_name", "category", "price", "weight_kg"]:
    record_check("null_check", "products", col, products_bronze.height, null_counts[col][0], "bronze")
record_check("null_check", "products", "product_id", products_bronze.height, rej_null.height, "bronze")
record_check("duplicate_check", "products", "product_id", products_bronze.height, rej_dup.height, "bronze")

record_check("cast_check", "products", "price", products_bronze.height, cast_price, "silver")
record_check("cast_check", "products", "weight_kg", products_bronze.height, cast_weight, "silver")
record_check("range_check", "products", "price", products_bronze.height, neg_price, "silver")
record_check("accepted_values", "products", "category", products_bronze.height, rej_cat.height, "silver")

quality = pl.DataFrame(quality_rows)
write_parquet(quality, config.QUALITY_DIR / "products.parquet")
quality

check_name,table,column,records_checked,records_failed,pct_failed,stage,executed_at
str,str,str,i64,i64,f64,str,datetime[μs]
"""null_check""","""products""","""product_name""",100,0,0.0,"""bronze""",2026-07-10 08:45:29.850236
"""null_check""","""products""","""category""",100,4,4.0,"""bronze""",2026-07-10 08:45:29.850241
"""null_check""","""products""","""price""",100,0,0.0,"""bronze""",2026-07-10 08:45:29.850243
"""null_check""","""products""","""weight_kg""",100,2,2.0,"""bronze""",2026-07-10 08:45:29.850244
"""null_check""","""products""","""product_id""",100,0,0.0,"""bronze""",2026-07-10 08:45:29.850260
"""duplicate_check""","""products""","""product_id""",100,0,0.0,"""bronze""",2026-07-10 08:45:29.850272
"""cast_check""","""products""","""price""",100,0,0.0,"""silver""",2026-07-10 08:45:29.850281
"""cast_check""","""products""","""weight_kg""",100,0,0.0,"""silver""",2026-07-10 08:45:29.850289
"""range_check""","""products""","""price""",100,2,2.0,"""silver""",2026-07-10 08:45:29.850297


In [14]:
try:
    Products.validate(products_silver, lazy=True)
    print("Products silver table passed")
except pa.errors.SchemaErrors as e:
    display(e.failure_cases)

Products silver table passed


Orders

In [15]:
orders = read_csv_raw("orders")
s_orders = orders.sample(fraction=0.1, seed=42)
s_orders.head(5)

order_id,customer_id,status,order_date,approved_at,delivered_at
str,str,str,str,str,str
"""ORD-000022""","""CUST-02511""","""delivered""","""2022-12-06 00:37:00""","""2022-12-07 06:37:00""","""2022-12-15 06:37:00"""
"""ORD-000028""","""CUST-03374""","""delivered""","""2023-11-07 21:35:00""","""2023-11-08 09:35:00""","""2023-11-16 09:35:00"""
"""ORD-000033""","""CUST-96659""","""canceled""","""2022-04-20 20:12:00""",null,null
"""ORD-000039""","""CUST-97638""","""shipped""","""2024-05-24 08:49:00""","""2024-05-24 19:49:00""",null
"""ORD-000074""","""CUST-03355""","""delivered""","""2023-03-25 05:47:00""","""2023-03-25 12:47:00""","""2023-03-26 12:47:00"""


In [16]:
class Orders(pa.DataFrameModel):
    order_id: str = pa.Field(nullable=False, unique=True)
    customer_id: str = pa.Field(nullable=False)
    status: str = pa.Field(nullable=True, isin=list(config.ORDER_STATUSES))
    order_date: datetime = pa.Field(nullable=True)
    approved_at: datetime = pa.Field(nullable=True)
    delivered_at: datetime = pa.Field(nullable=True)
    source_file: str = pa.Field(alias="_source_file", nullable=False)
    ingested_at: datetime = pa.Field(alias="_ingested_at", nullable=False)

    class Config:
        coerce = False
        strict = True

In [17]:
orders_bronze = s_orders.with_columns(
    pl.lit("orders.csv").alias("_source_file"),
    pl.lit(datetime.now()).alias("_ingested_at")
)
write_parquet(orders_bronze, config.BRONZE_DIR / "orders.parquet")

In [18]:
norm = read_csv_raw("orders")["status"].str.strip_chars().str.to_lowercase()
print("\nnormalizado (trim + minúsculas):")
print(norm.value_counts(sort=True))


normalizado (trim + minúsculas):
shape: (7, 2)
┌────────────┬───────┐
│ status     ┆ count │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ delivered  ┆ 8854  │
│ shipped    ┆ 2248  │
│ canceled   ┆ 1510  │
│ processing ┆ 1505  │
│ returned   ┆ 765   │
│ unknown    ┆ 59    │
│ null       ┆ 59    │
└────────────┴───────┘


In [19]:
orders = strip_strings(orders_bronze, ["order_id", "customer_id", "status"])
orders = lowercase(orders, "status")
orders = parse_datetime(orders, "order_date")
orders = parse_datetime(orders, "approved_at")
orders = parse_datetime(orders, "delivered_at")
orders, rej_status = nullify_not_in_set(orders, "status", config.ORDER_STATUSES)

valid_customers = read_csv_raw("customers")["customer_id"]
orders, rej_orphan = split_orphans(orders, "customer_id", valid_customers)

orders, rej_null = drop_null_keys(orders, "order_id")
orders, rej_dup = deduplicate_by_key(orders, "order_id", "order_date")

orders_silver = orders
orders_rejected = pl.concat([
    rej_status.with_columns(pl.lit("invalid_status").alias("reject_reason")),
    rej_orphan.with_columns(pl.lit("orphan_customer").alias("reject_reason")),
    rej_null.with_columns(pl.lit("null_order_id").alias("reject_reason")),
    rej_dup.with_columns(pl.lit("duplicate_order_id").alias("reject_reason")),
], how="diagonal")

write_parquet(orders_silver, config.SILVER_DIR / "orders.parquet")
write_parquet(orders_rejected, config.QUARANTINE_DIR / "orders.parquet")

In [20]:
quality_rows = []
def record_check(check_name, table, column, checked, failed, stage):
    quality_rows.append({
        "check_name": check_name, "table": table, "column": column,
        "records_checked": checked, "records_failed": failed,
        "pct_failed": round(100 * failed / checked, 2) if checked else 0.0,
        "stage": stage, "executed_at": datetime.now(),
    })

null_counts = orders_bronze.select(["status", "order_date", "approved_at", "delivered_at"]).null_count()
for col in ["status", "order_date", "approved_at", "delivered_at"]:
    record_check("null_check", "orders", col, orders_bronze.height, null_counts[col][0], "bronze")
record_check("null_check", "orders", "order_id", orders_bronze.height, rej_null.height, "bronze")
record_check("duplicate_check", "orders", "order_id", orders_bronze.height, rej_dup.height, "bronze")

record_check("accepted_values", "orders", "status", orders_bronze.height, rej_status.height, "silver")
record_check("referential_integrity", "orders", "customer_id", orders_bronze.height, rej_orphan.height, "silver")

quality = pl.DataFrame(quality_rows)
write_parquet(quality, config.QUALITY_DIR / "orders.parquet")
quality

check_name,table,column,records_checked,records_failed,pct_failed,stage,executed_at
str,str,str,i64,i64,f64,str,datetime[μs]
"""null_check""","""orders""","""status""",1500,3,0.2,"""bronze""",2026-07-10 08:45:29.891222
"""null_check""","""orders""","""order_date""",1500,11,0.73,"""bronze""",2026-07-10 08:45:29.891226
"""null_check""","""orders""","""approved_at""",1500,141,9.4,"""bronze""",2026-07-10 08:45:29.891228
"""null_check""","""orders""","""delivered_at""",1500,607,40.47,"""bronze""",2026-07-10 08:45:29.891229
"""null_check""","""orders""","""order_id""",1500,0,0.0,"""bronze""",2026-07-10 08:45:29.891245
"""duplicate_check""","""orders""","""order_id""",1500,1,0.07,"""bronze""",2026-07-10 08:45:29.891256
"""accepted_values""","""orders""","""status""",1500,3,0.2,"""silver""",2026-07-10 08:45:29.891265
"""referential_integrity""","""orders""","""customer_id""",1500,29,1.93,"""silver""",2026-07-10 08:45:29.891273


In [21]:
try:
    Orders.validate(orders_silver, lazy=True)
    print("Orders silver table passed")
except pa.errors.SchemaErrors as e:
    display(e.failure_cases)

Orders silver table passed


Order Items

In [22]:
order_items = read_csv_raw("order_items")
s_order_items = order_items.sample(fraction=0.1, seed=42)
s_order_items.head(5)

item_id,order_id,product_id,quantity,unit_price,freight_value
str,str,str,str,str,str
"""ITEM-000001""","""ORD-000383""","""PROD-0627""","""5""","""2534.91""","""140.05"""
"""ITEM-000006""","""ORD-009322""","""PROD-0249""","""3""","""1552.64""","""133.59"""
"""ITEM-000011""","""ORD-014452""","""PROD-0184""","""1""","""1942.39""","""97.22"""
"""ITEM-000013""","""ORD-004111""","""PROD-0581""","""5""","""2713.74""","""121.39"""
"""ITEM-000027""","""ORD-007902""","""PROD-0574""","""5""","""34.32""","""118.71"""


In [ ]:
class OrderItems(pa.DataFrameModel):
    item_id: str = pa.Field(nullable=False, unique=True)
    order_id: str = pa.Field(nullable=False)
    product_id: str = pa.Field(nullable=False)
    quantity: int = pa.Field(nullable=False, gt=0)
    unit_price: float = pa.Field(nullable=False, ge=0)
    freight_value: float = pa.Field(nullable=True, ge=0)
    source_file: str = pa.Field(alias="_source_file", nullable=False)
    ingested_at: datetime = pa.Field(alias="_ingested_at", nullable=False)

    class Config:
        coerce = False
        strict = True

In [24]:
order_items_bronze = s_order_items.with_columns(
    pl.lit("order_items.csv").alias("_source_file"),
    pl.lit(datetime.now()).alias("_ingested_at")
)
write_parquet(order_items_bronze, config.BRONZE_DIR / "order_items.parquet")